# 长鑫科技合理估值分析（四）第7节：进阶回归、事件研究、预测

**原则**：每一张图、每一个回归系数均来自 `data/` 已下载的**官方/交易所披露数据**，不手工编造。

| 分析 | 官方/权威数据源 | 本地文件 |
|------|-----------------|----------|
| 长鑫财务 | 上交所招股说明书（申报稿 2025-12-30） | `data/ipo/cxkj_financials.csv` |
| IPO 事件日 | 安徽证监局辅导备案、上交所受理日期 | `cxkj_ipo_tutor.csv`, `cxkj_register.csv` |
| A 股行情 | 交易所日线（akshare→新浪） | `data/stock/*.csv` |
| 指数 | 沪深300、科创50 | `data/index/*.csv` |
| 行业指数 | 同花顺半导体指数 | `data/industry/semiconductor_index_ths.csv` |
| 美光财报 | SEC 年报汇总（akshare 东方财富） | `data/finance/mu_annual_us.csv` |
| 同业财务 | 同花顺财务摘要 | `data/finance/finance_ths_*.csv` |

运行 `python scripts/advanced_analysis.py` 可复现全部输出至 `output/`。


In [ ]:
from pathlib import Path
import os, sys
ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "scripts"))
from plot_style import setup_theme
setup_theme()
DATA, CLEAN, OUTPUT = ROOT / "data", ROOT / "clean", ROOT / "output"
print("项目根目录:", ROOT)


## 7.1 CAPM 与多因子回归（日频行情 + 官方指数）

In [ ]:
import pandas as pd
from advanced_analysis import run_section7
# 执行完整流水线（生成 fig7_*.png，中文由 plot_style 保证）
run_section7()

capm = pd.read_csv(OUTPUT / "regression_capm_summary.csv")
display(capm[capm["benchmark"].isin(["沪深300", "科创50"])].sort_values(["name", "benchmark"]))


**图形** `fig7_capm_beta_dual_benchmark.png`：蓝柱为相对**沪深300**（中证指数）的 Beta，紫柱为相对**科创50**。

**数据结论**（来自 `regression_capm_summary.csv`）：
- 兆易创新、澜起科技、北京君正 Beta 约 **1.31—1.43**，且 `beta_pvalue` 极小 → 与市场的协动**统计显著**。
- 存储股 Beta 高于中芯国际（约 1.04），说明 long 存储板块时**系统性风险敞口更大**，IPO 定价窗口若遇市场下行，折价压力更高。

**散点图** `fig7_capm_scatter_603986.png`：以兆易创新日收益对沪深300 日收益作 OLS，直观展示拟合线与 R²。


## 7.2 事件研究（官方 IPO 节点 + 市场模型）

In [ ]:
events = pd.read_csv(OUTPUT / "event_study_events.csv")
tests = pd.read_csv(OUTPUT / "event_study_significance.csv")
display(events)
display(tests)


**方法**（标准事件研究）：
- **事件日**：辅导完成备案 **2025-07-07**（`cxkj_ipo_tutor.csv`）；科创板受理 **2025-12-30**（`cxkj_register.csv`）。
- **估计窗**：事件日前第 120 至 21 个交易日；**事件窗**：[-10, +10]。
- **市场模型**：$R_{i,t}=\alpha+\beta R_{m,t}+\varepsilon_t$，$R_m$ 为沪深300 日收益。

**图形** `fig7_event_study_car_detailed.png`：半导体指数、存储同业等权组合的累计异常收益 CAR。

**数据结论**（`event_study_significance.csv`）：
- **受理日**（2025-12-30）：半导体指数事件窗末端 CAR 约 **+6.45%**，但 AR 均值 t 检验 **p≈0.15>0.05** → 异常收益**不显著**。
- **辅导完成日**（2025-07-07）：CAR 约 **-0.5%**，p≈0.91 → 无显著市场反应。

**解读**：长鑫 IPO 关键节点对半导体板块**未产生统计显著的异常波动**，信息或已提前消化，或被行业整体波动淹没。


## 7.3 招股书财务回归 + 营收预测（仅上交所披露数据）

In [ ]:
cxkj_reg = pd.read_csv(OUTPUT / "regression_cxkj_summary.csv")
fc = pd.read_csv(OUTPUT / "revenue_forecast_ols_ci.csv")
display(cxkj_reg)
display(fc)


**模型 A：营收 ~ 年份**（2022—2024 招股书，`fig7_revenue_ols_forecast.png`）
- OLS 年斜率约 **+79.5 亿元/年**（`regression_cxkj_summary.csv`）；因仅 3 个年度点，p>0.05，**趋势显著性不足**，预测区间较宽。
- 2025—2027 点预测与 95% CI 见 `revenue_forecast_ols_ci.csv` → 仅作**敏感性参考**，不能替代产能/DRAM 价格情景。

**模型 B：归母净利润 ~ 营业收入**（`fig7_profit_revenue_regression.png`）
- 斜率为正但 R²≈0.32、p≈0.62 → **亏损收窄与收入扩张有关，但三年样本无法稳定估计盈利拐点**。

**模型 C：长鑫营收 vs 美光营收**（`fig7_cxmt_mu_revenue_regression.png`，MU 年报 `OPERATE_INCOME`）
- 展示国产 DRAM 龙头与全球龙头的**周期协同**；系数受样本长度限制，用于说明**行业景气联动**，非因果预测。


## 7.4 同业 PS 对数回归 → 长鑫隐含市值

In [ ]:
ps_panel = pd.read_csv(CLEAN / "peer_ps_panel.csv")
ps_res = pd.read_csv(OUTPUT / "implied_valuation_ps_regression.csv")
display(ps_panel.tail(10))
display(ps_res)


**方法**：对可比公司（兆易创新、澜起科技、北京君正、中芯国际）各年观测，
$\ln(\text{市值}) = a + b \ln(\text{营业总收入}) + \varepsilon$。

- **市值** = 年末最后一个交易日收盘价 × 流通股（`data/stock` 行情）。
- **收入** = 同花顺财务摘要 `营业总收入`（`finance_ths_*.csv`）。

**图形** `fig7_peer_ps_regression.png`：散点为同业历年观测，红星为长鑫 2024 收入对应的**回归隐含市值**。

**数据结论**（`implied_valuation_ps_regression.csv`）：
- 2024 收入 **241.78 亿元** → 模型隐含市值约 **1636 亿元**，隐含 PS 约 **6.77×**。
- 回归 R² 较低（≈0.08）→ 同业估值分化大，**模型仅作区间锚定**。
- 与一级市场报道 **1282—1584 亿元** 相比：模型点估计略高，落在同一数量级，支持「**高 PS 成长溢价**」叙事，但需下调 R² 不确定性。

---

完整文字报告：`output/section7_conclusions.md`


In [ ]:
# 展示全部第7节图表路径
for p in sorted(OUTPUT.glob("fig7_*.png")):
    print(p.name)
